# Maria's HW2: LLM Paraphrase Encoder Subgroup Analysis
**Изолированный эксперимент.** Не изменяет файлы Анастасии или Дилары. Все артефакты сохраняются в Google Drive.

In [2]:
# 1. GPU и Google Drive (HF_HOME направлен в Drive ДО загрузки модели)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os
from google.colab import drive
drive.mount('/content/drive')

os.environ['HF_HOME'] = '/content/drive/MyDrive/huggingface_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

import pathlib
WORK = pathlib.Path('/content/drive/MyDrive/far-trans-hw2/maria_outputs')
WORK.mkdir(parents=True, exist_ok=True)
print('Work directory:', WORK)

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
Mounted at /content/drive
Work directory: /content/drive/MyDrive/far-trans-hw2/maria_outputs


In [3]:
# 2. Репозиторий курса (строго зафиксированный коммит)
COURSE_COMMIT = 'f6850861ea20a51e1fb018ee4e5f840c2ac759e0'
!rm -rf /content/RecSys-LLMs && git clone -q https://github.com/dryjins/RecSys-LLMs.git /content/RecSys-LLMs
!git -C /content/RecSys-LLMs checkout -q {COURSE_COMMIT}
COURSE = '/content/RecSys-LLMs/project'

!git -C /content/RecSys-LLMs log -1 --format='Course commit: %H'

Course commit: f6850861ea20a51e1fb018ee4e5f840c2ac759e0


In [8]:
# 3. Командный репозиторий (ветка Марии)
!rm -rf /content/team_repo && git clone -q https://github.com/getmanyan/far-trans-group7.git /content/team_repo
!git -C /content/team_repo fetch origin
!git -C /content/team_repo checkout -q maria/hw2-paraphrase-encoder

TEAM_ROOT = '/content/team_repo'
MARIA_CODE = '/content/team_repo/maria_hw2'
print('Maria code ready at:', MARIA_CODE)
!git -C /content/team_repo log -1 --format='Team commit: %H (%s)'

Maria code ready at: /content/team_repo/maria_hw2
Team commit: c54e30af08de80ee7b8b5767cde41a6a191be21d (Add files via upload)


In [5]:
# 4. Данные
import zipfile
ZIP = pathlib.Path('/content/drive/MyDrive/far-trans-hw2/FAR-Trans.zip')
if not ZIP.exists():
    from google.colab import files
    print("Please upload FAR-Trans.zip")
    up = files.upload()
    ZIP.write_bytes(up[next(iter(up))])

with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/data')
DATA = '/content/data/FAR-Trans'
!ls -la {DATA}

Please upload FAR-Trans.zip


Saving FAR-Trans.zip to FAR-Trans.zip
total 59492
drwxr-xr-x 2 root root     4096 Nov 20 11:42 .
drwxr-xr-x 3 root root     4096 Nov 20 11:42 ..
-rw-r--r-- 1 root root    83851 Nov 20 11:42 asset_information.csv
-rw-r--r-- 1 root root 22668996 Nov 20 11:42 close_prices.csv
-rw-r--r-- 1 root root  2408798 Nov 20 11:42 customer_information.csv
-rw-r--r-- 1 root root    56622 Nov 20 11:42 limit_prices.csv
-rw-r--r-- 1 root root    11771 Nov 20 11:42 markets.csv
-rw-r--r-- 1 root root     7216 Nov 20 11:42 questionnaires.csv
-rw-r--r-- 1 root root     7135 Nov 20 11:42 README.txt
-rw-r--r-- 1 root root 35651778 Nov 20 11:42 transactions.csv


In [6]:
# 5. Зависимости
!pip -q install transformers==4.57.6 huggingface_hub pandas
import torch
print('torch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
torch: 2.14.1+cu130 | GPU: Tesla T4


In [13]:
# 6. Unit-тесты логики подгрупп Марии
!cd {MARIA_CODE} && python -m unittest test_maria.py -v

test_missing_method_raises_error (test_maria.TestMariaAnalysis.test_missing_method_raises_error) ... ok
test_subgroup_aggregation (test_maria.TestMariaAnalysis.test_subgroup_aggregation) ... ok
test_user_outside_manifest_excluded (test_maria.TestMariaAnalysis.test_user_outside_manifest_excluded) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.682s

OK


In [14]:
# 7. Загрузка модели (кэшируется в HF_HOME на Google Drive)
from huggingface_hub import snapshot_download
MODEL_ID = 'Qwen/Qwen3-1.7B'
REVISION = '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'
MODEL_PATH = snapshot_download(MODEL_ID, revision=REVISION)
print('Model path:', MODEL_PATH)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

merges.txt: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (5 outputs left out)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Model path: /content/drive/MyDrive/huggingface_cache/hub/models--Qwen--Qwen3-1.7B/snapshots/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e


In [15]:
# 8. Baseline (4 метода курса)
print("Running baselines...")
!cd {COURSE}/far-trans && python starter.py run --data-dir {DATA} --output-dir {WORK}/baseline

import json
baseline_metrics = WORK / 'baseline' / 'metrics.json'
if baseline_metrics.exists():
    with open(baseline_metrics, 'r') as f:
        bm = json.load(f)
    print("✅ Baseline metrics loaded. Methods:", list(bm['methods'].keys()))
else:
    raise FileNotFoundError("❌ Baseline metrics not found.")

Running baselines...
Run summary (console values rounded to 4 decimals; metrics.json keeps full precision)
method              users  nonempty truth  empty truth  nDCG@10 (all)  nDCG@10 (nonempty)
------------------  -----  --------------  -----------  -------------  ------------------
cbf                 100                11           89         0.0201              0.1827
latest_buy          100                11           89         0.0277              0.2520
latest_buy_or_sell  100                11           89         0.0271              0.2464
popularity          100                11           89         0.0286              0.2599
protocol_hash 50efd4aeb415 (prefix; full value in protocol.json and metrics.json)
excluded_rows {"reconstructed_initial_position_rows": 38019}
custom_failures 0
Offline purchase relevance, not profitability or asset suitability.
✅ Baseline metrics loaded. Methods: ['cbf', 'latest_buy', 'latest_buy_or_sell', 'popularity']


In [16]:
# 9. Эксперимент Марии (передаём MODEL_ID и REVISION, а не локальный путь)
print("Running Maria's experiment via Anastasia's wrapper...")
!cd {MARIA_CODE} && python run_maria_experiment.py \
    --team-root {TEAM_ROOT} \
    --course-dir {COURSE} \
    --data-dir {DATA} \
    --output-dir {WORK}/maria_llmrec \
    --model {MODEL_ID} \
    --revision {REVISION}

Running Maria's experiment via Anastasia's wrapper...
Executing: /usr/bin/python3 /content/team_repo/run_course_encoder.py --course-dir /content/RecSys-LLMs/project --data-dir /content/data/FAR-Trans --output-dir /content/drive/MyDrive/far-trans-hw2/maria_outputs/maria_llmrec --model Qwen/Qwen3-1.7B --revision 70d244cc86ccca08cf5af4e1e306ecf908b1ad5e --device cuda --dtype auto --on-invalid fallback
Attribution: Using Anastasia's run_course_encoder.py without modification.
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1795175363.869647    3913 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795175368.273141    3913 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Loading checkpoint shards: 100% 2/2 [00:13<00:

In [17]:
# 10. Проверка совпадения protocol параметров и baseline метрик
import json

baseline_protocol = json.load(open(WORK / 'baseline' / 'protocol.json'))
baseline_metrics = json.load(open(WORK / 'baseline' / 'metrics.json'))

maria_manifest = json.load(open(WORK / 'maria_llmrec' / 'run_manifest.json'))
maria_metrics = json.load(open(WORK / 'maria_llmrec' / 'metrics.json'))

assert maria_manifest['runtime']['model'] == MODEL_ID, f"Model mismatch: {maria_manifest['runtime']['model']} vs {MODEL_ID}"
assert maria_manifest['runtime']['requested_revision'] == REVISION, f"Revision mismatch: {maria_manifest['runtime']['requested_revision']} vs {REVISION}"

protocol_keys = ['users', 'cutoff', 'end_exclusive', 'candidate_count']
for key in protocol_keys:
    assert baseline_protocol[key] == maria_manifest[key], f"Mismatch in {key}: {baseline_protocol[key]} vs {maria_manifest[key]}"

assert baseline_protocol['source_sha256'] == maria_manifest['data_sha256'], f"Data hash mismatch: {baseline_protocol['source_sha256']} vs {maria_manifest['data_sha256']}"

baseline_methods = ['popularity', 'cbf', 'latest_buy', 'latest_buy_or_sell']
for method in baseline_methods:
    bm = baseline_metrics['methods'][method]
    mm = maria_metrics[method]

    assert bm['users'] == mm['users'], f"Users mismatch for {method}"
    assert bm['nonempty_truth_users'] == mm['nonempty_truth_users'], f"Nonempty truth users mismatch for {method}"
    assert bm['ndcg_at_10'] == mm['ndcg_at_10'], f"nDCG@10 mismatch for {method}: {bm['ndcg_at_10']} vs {mm['ndcg_at_10']}"
    assert bm['ndcg_at_10_nonempty'] == mm['ndcg_at_10_nonempty'], f"nDCG@10 nonempty mismatch for {method}: {bm['ndcg_at_10_nonempty']} vs {mm['ndcg_at_10_nonempty']}"

print("✅ All protocol parameters match between baseline and maria_llmrec")
print(f"✅ Model in manifest: {maria_manifest['runtime']['model']}, Revision: {maria_manifest['runtime']['requested_revision']}")
print(f"✅ All 4 baseline metrics match exactly")

✅ All protocol parameters match between baseline and maria_llmrec
✅ Model in manifest: Qwen/Qwen3-1.7B, Revision: 70d244cc86ccca08cf5af4e1e306ecf908b1ad5e
✅ All 4 baseline metrics match exactly


In [18]:
# 11. Анализ подгрупп Марии
print("Running subgroup analysis...")
!cd {MARIA_CODE} && python analyze_maria.py \
    --data-dir {DATA} \
    --course-dir {COURSE} \
    --run-dir {WORK}/maria_llmrec \
    --out {WORK}/results_maria.md

Running subgroup analysis...
Analysis complete. Results saved to /content/drive/MyDrive/far-trans-hw2/maria_outputs/results_maria.md
# Maria's HW2 Subgroup Analysis Results

Total target users evaluated: 100
Users with non-empty ground truth: 11

### Validation
- Length groups: {'len_1': 40, 'len_2_3': 23, 'len_4_plus': 37}
- Metadata groups: {'meta_both': 46, 'meta_none': 22, 'meta_mixed': 32}
- Data leakage check: ✅ PASSED

### 1. History Length Subgroups
| Group | Total | Non-Empty | Method | nDCG@10 All | nDCG@10 Non-Empty |
|---|---:|---:|---|---:|---:|
| len_1 | 40 | 0 | original_text_cbf | 0.0000 | 0.0000 |
| len_1 | 40 | 0 | **llmrec_encoder** | 0.0000 | 0.0000 |
| len_2_3 | 23 | 1 | original_text_cbf | 0.0000 | 0.0000 |
| len_2_3 | 23 | 1 | **llmrec_encoder** | 0.0000 | 0.0000 |
| len_4_plus | 37 | 10 | original_text_cbf | 0.0636 | 0.2354 |
| len_4_plus | 37 | 10 | **llmrec_encoder** | 0.0638 | 0.2360 |

### 2. Metadata Completeness Subgroups
| Group | Total | Non-Empty | Meth

In [19]:
# 12. Вывод результатов
from IPython.display import Markdown
try:
    display(Markdown(open(f'{WORK}/results_maria.md').read()))
except FileNotFoundError:
    print("❌ Results file not found. Check previous cell output for errors.")

# Maria's HW2 Subgroup Analysis Results

Total target users evaluated: 100
Users with non-empty ground truth: 11

### Validation
- Length groups: {'len_1': 40, 'len_2_3': 23, 'len_4_plus': 37}
- Metadata groups: {'meta_both': 46, 'meta_none': 22, 'meta_mixed': 32}
- Data leakage check: ✅ PASSED

### 1. History Length Subgroups
| Group | Total | Non-Empty | Method | nDCG@10 All | nDCG@10 Non-Empty |
|---|---:|---:|---|---:|---:|
| len_1 | 40 | 0 | original_text_cbf | 0.0000 | 0.0000 |
| len_1 | 40 | 0 | **llmrec_encoder** | 0.0000 | 0.0000 |
| len_2_3 | 23 | 1 | original_text_cbf | 0.0000 | 0.0000 |
| len_2_3 | 23 | 1 | **llmrec_encoder** | 0.0000 | 0.0000 |
| len_4_plus | 37 | 10 | original_text_cbf | 0.0636 | 0.2354 |
| len_4_plus | 37 | 10 | **llmrec_encoder** | 0.0638 | 0.2360 |

### 2. Metadata Completeness Subgroups
| Group | Total | Non-Empty | Method | nDCG@10 All | nDCG@10 Non-Empty |
|---|---:|---:|---|---:|---:|
| meta_both | 46 | 3 | original_text_cbf | 0.0067 | 0.1022 |
| meta_both | 46 | 3 | **llmrec_encoder** | 0.0067 | 0.1022 |
| meta_none | 22 | 0 | original_text_cbf | 0.0000 | 0.0000 |
| meta_none | 22 | 0 | **llmrec_encoder** | 0.0000 | 0.0000 |
| meta_mixed | 32 | 8 | original_text_cbf | 0.0640 | 0.2560 |
| meta_mixed | 32 | 8 | **llmrec_encoder** | 0.0642 | 0.2566 |

In [21]:
# 13. Упаковка артефактов (без датасетов и весов модели)
!cd {WORK} && rm -f maria_hw2_outputs.zip && zip -qr maria_hw2_outputs.zip . -x "*.zip" -x "*FAR-Trans*" -x "*huggingface_cache*"
from google.colab import files
files.download(f'{WORK}/maria_hw2_outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
import os
import shutil

src = "/content/team_repo"
dst = os.path.join(src, "maria_hw2")
os.makedirs(dst, exist_ok=True)

files = [
    "README.md",
    "run_maria_experiment.py",
    "analyze_maria.py",
    "test_maria.py",
    "hw2_maria_colab.ipynb",
]

for name in files:
    shutil.copy2(
        os.path.join(src, "maria_hw2:" + name),
        os.path.join(dst, name)
    )

print("Файлы в правильной папке:")
print(os.listdir(dst))

Файлы в правильной папке:
['analyze_maria.py', 'run_maria_experiment.py', 'README.md', 'hw2_maria_colab.ipynb', 'test_maria.py']


In [25]:
import zipfile

path = "/content/drive/MyDrive/far-trans-hw2/maria_outputs/maria_hw2_outputs.zip"

with zipfile.ZipFile(path) as z:
    files = [x for x in z.namelist() if not x.endswith("/")]
    print("Всего файлов:", len(files))
    print("\n".join(files[:15]))
    print("\n...")

    keywords = ["metric", "manifest", "protocol", "result", "baseline",
                "subgroup", "encoder", "predict", "report", "readme"]

    print("\nПроверка важных файлов:")
    for word in keywords:
        matches = [x for x in files if word in x.lower()]
        print(f"{word}: {len(matches)}")
        for x in matches[:3]:
            print("  ", x)

Всего файлов: 715
baseline/protocol.json
baseline/contexts.jsonl
baseline/predictions.csv
baseline/per_user.csv
baseline/metrics.json
maria_llmrec/encoder_prompts.jsonl
maria_llmrec/generation_cache/ee7133b855d3c4d472ce2f3809385b058fefebb32f7ffb06fedc01da0554fd0e.json
maria_llmrec/generation_cache/f2677b8ca3bc159f13c24a7ddd2d0ce2c3c757587536d388ed6972140b877bf0.json
maria_llmrec/generation_cache/9ac87c687cbc2174d5f767afeacfbdadcf767866b6e336ed653437981042d82d.json
maria_llmrec/generation_cache/06bb900df2d79589b075cd6669776ce9ead5b239d4f10d27b3c24ea6d5be0b9a.json
maria_llmrec/generation_cache/a8c8b6c342192bb2567686aece0e93baa7001de96c230f312bd570477219c9e6.json
maria_llmrec/generation_cache/48d8cce1832d5df8534e930029108f2c6fc6ee72d01c07af031da4a85f66d237.json
maria_llmrec/generation_cache/65c4ead5c5fea75166424a47389235b431b5ca8af0eeeee17b8a3b368a3a4644.json
maria_llmrec/generation_cache/f61c2d5ea65cef58c4d66ba95228dd2217038f537529de64581779596d2870d7.json
maria_llmrec/generation_cache/b

In [26]:
import zipfile

p = "/content/drive/MyDrive/far-trans-hw2/maria_outputs/maria_hw2_outputs.zip"

required = [
    "baseline/metrics.json",
    "baseline/protocol.json",
    "baseline/per_user.csv",
    "baseline/predictions.csv",
    "maria_llmrec/metrics.json",
    "maria_llmrec/run_manifest.json",
    "maria_llmrec/per_user.json",
    "maria_llmrec/predictions.csv",
    "maria_llmrec/encoder_prompts.jsonl",
    "results_maria.md"
]

with zipfile.ZipFile(p) as z:
    names = set(z.namelist())
    for f in required:
        print(("OK      " if f in names else "MISSING "), f)

OK       baseline/metrics.json
OK       baseline/protocol.json
OK       baseline/per_user.csv
OK       baseline/predictions.csv
OK       maria_llmrec/metrics.json
OK       maria_llmrec/run_manifest.json
OK       maria_llmrec/per_user.json
OK       maria_llmrec/predictions.csv
OK       maria_llmrec/encoder_prompts.jsonl
OK       results_maria.md


In [27]:
import json
p = "/content/drive/MyDrive/far-trans-hw2/maria_outputs/maria_llmrec/metrics.json"
print(json.dumps(json.load(open(p)), indent=2))

{
  "cbf": {
    "users": 100,
    "nonempty_truth_users": 11,
    "empty_truth_users": 89,
    "ndcg_at_10": 0.020100011736078574,
    "ndcg_at_10_nonempty": 0.18272737941889614
  },
  "latest_buy": {
    "users": 100,
    "nonempty_truth_users": 11,
    "empty_truth_users": 89,
    "ndcg_at_10": 0.027722649787649066,
    "ndcg_at_10_nonempty": 0.2520240889786279
  },
  "latest_buy_or_sell": {
    "users": 100,
    "nonempty_truth_users": 11,
    "empty_truth_users": 89,
    "ndcg_at_10": 0.027107048693904026,
    "ndcg_at_10_nonempty": 0.2464277153991275
  },
  "llmrec_encoder": {
    "users": 100,
    "nonempty_truth_users": 11,
    "empty_truth_users": 89,
    "ndcg_at_10": 0.023596154532097634,
    "ndcg_at_10_nonempty": 0.21451049574634212
  },
  "original_text_cbf": {
    "users": 100,
    "nonempty_truth_users": 11,
    "empty_truth_users": 89,
    "ndcg_at_10": 0.023542084151329593,
    "ndcg_at_10_nonempty": 0.21401894683026904
  },
  "popularity": {
    "users": 100,
    "no